# MicroCLIP — Colab A100 Training

Runs any config from the ablation matrix on a Colab A100.

**Setup:** Runtime → Change runtime type → **A100 GPU**.

**Storage layout (Drive-space friendly):**

| What | Where | Size |
|---|---|---|
| COCO images | local SSD (`/content`) | ~20 GB, never touches Drive |
| Checkpoints while training | local SSD (`runs/`) | written every 500 steps + every epoch |
| Checkpoint mirror | Drive `MyDrive/microclip/runs/<run>/` | `last_a.pt` + `last_b.pt` + `best.pt` ≈ **0.5 GB per run** |
| Tokenizer | Drive `MyDrive/microclip/artifacts/` | ~1 MB |

Why not write checkpoints straight to Drive: the trainer replaces the ~214 MB `last.pt`
40–90 times per run. Files deleted or replaced through the Colab Drive mount end up in
Drive **Trash**, which still counts against quota — a single run can fill a 15 GB Drive.
Instead, a background process copies checkpoints to Drive every `SYNC_EVERY_MIN` minutes,
overwriting two fixed slot files in place (no deletes → nothing goes to Trash). If a
copy is torn by preemption, the other slot still holds the previous good checkpoint.

**Preemption:** re-run the whole notebook. The newest readable Drive slot is restored to
local disk and training auto-resumes from it (you lose at most ~`SYNC_EVERY_MIN` minutes).

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > A100"
name = torch.cuda.get_device_name(0)
print(name, "| bf16:", torch.cuda.is_bf16_supported())
if "A100" not in name:
    print("WARNING: not an A100 — configs assume Ampere+ (bf16 AMP, batch 512)")

NVIDIA A100-SXM4-40GB | bf16: True


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import os

PERSIST = "/content/drive/MyDrive/microclip"
os.makedirs(f"{PERSIST}/runs", exist_ok=True)
os.makedirs(f"{PERSIST}/artifacts/tokenizer", exist_ok=True)

# Budget ~0.5 GB of free Drive per run you want to keep. If Drive is near full,
# empty Drive Trash first (drive.google.com → Trash) — it counts against quota.
!df -h /content/drive | tail -1
!du -sh {PERSIST}

Mounted at /content/drive
drive            15G   11G  4.2G  73% /content/drive
5.1G	/content/drive/MyDrive/microclip


In [ ]:
import os, torch

root = "/content/drive/MyDrive/microclip/runs"
for name in sorted(os.listdir(root)):
    d = f"{root}/{name}"
    if not os.path.isdir(d): continue
    cands = [f"{d}/{fn}" for fn in ("last_a.pt","last_b.pt") if os.path.exists(f"{d}/{fn}")]
    newest = max(cands, key=os.path.getmtime)   # en son yazılanı seç, tek yükleme
    s = torch.load(newest, map_location="cpu", weights_only=False)
    ep = s.get("epoch","?"); st = s.get("global_step","?")
    bv = s.get("best_val", float("nan"))
    print(f"{name:16s} {os.path.basename(newest):9s} epoch={ep:>3} step={st:>6} "
          f"best_val={bv:.4f} keys={sorted(s.keys())[:6]}")
    del s

# best.pt tek örnek şema kontrolü:
bp = torch.load(f"{root}/sigmoid_b512/best.pt", map_location="cpu", weights_only=False)
print("best.pt:", type(bp).__name__,
      (sorted(bp.keys())[:6] if isinstance(bp, dict) else ""))

abl_init_he      last_b.pt epoch= 10 step=  2310 best_val=5.4878 keys=['batch_idx', 'best_val', 'epoch', 'global_step', 'model', 'optimizer']
abl_init_xavier  last_b.pt epoch= 10 step=  2310 best_val=5.3290 keys=['batch_idx', 'best_val', 'epoch', 'global_step', 'model', 'optimizer']
abl_lr_constant  last_b.pt epoch= 10 step=  2310 best_val=5.4488 keys=['batch_idx', 'best_val', 'epoch', 'global_step', 'model', 'optimizer']
abl_sgd          last_b.pt epoch= 10 step=  2310 best_val=7.2136 keys=['batch_idx', 'best_val', 'epoch', 'global_step', 'model', 'optimizer']
abl_vit_tiny     last_b.pt epoch= 30 step=  6930 best_val=5.1428 keys=['batch_idx', 'best_val', 'epoch', 'global_step', 'model', 'optimizer']
sigmoid_b128     last_b.pt epoch= 30 step= 27720 best_val=3.7937 keys=['batch_idx', 'best_val', 'epoch', 'global_step', 'model', 'optimizer']
sigmoid_b256     last_b.pt epoch= 30 step= 13860 best_val=4.2352 keys=['batch_idx', 'best_val', 'epoch', 'global_step', 'model', 'optimizer']
sigmoi

In [ ]:
%cd /content
!git clone https://github.com/umutonuryasar/microclip.git 2>/dev/null || git -C microclip pull
%cd /content/microclip
!pip -q install -e .

# pip -e registers src/ via a .pth file, which a running kernel never re-reads —
# without this, `import microclip` in this notebook fails until a runtime restart.
import sys
if "/content/microclip/src" not in sys.path:
    sys.path.insert(0, "/content/microclip/src")

/content
/content/microclip
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for microclip (pyproject.toml) ... done


In [ ]:
import os

# Tokenizer is tiny -> lives on Drive so it is trained once.
if not os.path.islink("artifacts"):
    os.system("rm -rf artifacts")
    os.symlink(f"{PERSIST}/artifacts", "artifacts")

# Checkpoints go to the LOCAL SSD (fast, no Drive Trash churn); the sync process
# below mirrors them to Drive. Undo the old runs/ -> Drive symlink if present.
if os.path.islink("runs"):
    os.unlink("runs")
os.makedirs("runs", exist_ok=True)
print("artifacts/ ->", os.readlink("artifacts"))
print("runs/ is local:", os.path.abspath("runs"))

artifacts/ -> /content/drive/MyDrive/microclip/artifacts
runs/ is local: /content/microclip/runs


In [ ]:
%%bash
# COCO 2017 to local SSD (~20 GB unzipped, ~40 GB peak while unzipping train2017).
df -h /content | tail -1
mkdir -p data/coco && cd data/coco
for z in train2017 val2017; do
  if [ ! -d $z ]; then
    wget -q -c http://images.cocodataset.org/zips/$z.zip
    unzip -q $z.zip && rm $z.zip
  fi
done
if [ ! -d annotations ]; then
  wget -q -c http://images.cocodataset.org/annotations/annotations_trainval2017.zip
  unzip -q annotations_trainval2017.zip && rm annotations_trainval2017.zip
fi
du -sh *

overlay         236G   48G  189G  21% /
796M	annotations
19G	train2017
788M	val2017


In [ ]:
import os

# One-off; lands on Drive via the artifacts/ symlink.
if not os.path.exists("artifacts/tokenizer/bpe16k.json"):
    !python scripts/train_tokenizer.py --config configs/base.yml
else:
    print("tokenizer exists — skipping")

[00:00:00] Tokenize words                 ██████████████████ 36897    /    36897
[00:00:00] Count pairs                    ██████████████████ 36897    /    36897
[00:00:00] Compute merges                 ██████████████████ 15906    /    15906
Tokenizer saved to artifacts/tokenizer/bpe16k.json


In [ ]:
# Sanity check on the real data path (~few min on A100) before burning GPU hours.
# Writes only to local runs/smoke_*; cleaned up afterwards. Skip on later sessions.
!python scripts/smoke_test.py --config configs/base.yml && rm -rf runs/smoke_sigmoid runs/smoke_softmax


=== smoke [sigmoid]: 4 epochs x 15 steps ===
[sigmoid] loss 6.3786 -> 4.9516, val 5.0028 OK
[resume] epoch=4 step=60 best_val=inf
[sigmoid] resume OK at step 60

=== smoke [softmax]: 4 epochs x 15 steps ===
[softmax] loss 4.1103 -> 3.6904, val 3.8441 OK
[resume] epoch=4 step=60 best_val=inf
[softmax] resume OK at step 60

SMOKE TEST PASSED — safe to launch A100 runs.


In [ ]:
import os
from google.colab import userdata

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

import wandb
wandb.login()

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: umutonuryasar (umutonuryasar-independent) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Checkpointing (Drive-safe)

The next cell writes `ckpt_sync.py`: a background process that mirrors local
`runs/<run>/{last,best}.pt` to Drive, alternating two fixed slot files
(`last_a.pt`/`last_b.pt`) so no delete/rename ever hits Drive Trash. Cells A and B
below start/stop it automatically per run. Run the writefile cell once per session.


In [ ]:
%%writefile /content/ckpt_sync.py
"""Mirror runs/<run>/{last,best}.pt to Drive without filling Drive Trash.

last.pt alternates between two fixed slot files (last_a.pt / last_b.pt) and every
copy overwrites a slot in place — no delete/rename on Drive, so no Trash growth.
A copy torn by preemption only damages one slot; the other stays valid.
"""
import os
import shutil
import sys
import time

local, remote, interval, stop = sys.argv[1], sys.argv[2], int(sys.argv[3]), sys.argv[4]
os.makedirs(remote, exist_ok=True)
slots = [os.path.join(remote, "last_a.pt"), os.path.join(remote, "last_b.pt")]
mtime = lambda p: os.path.getmtime(p) if os.path.exists(p) else 0.0
slot = 0 if mtime(slots[0]) <= mtime(slots[1]) else 1  # overwrite the older slot first
seen = {}


def push(name):
    global slot
    src = os.path.join(local, name)
    if not os.path.exists(src):
        return
    # Open first: the trainer's os.replace() can't swap the file under us mid-copy.
    with open(src, "rb") as fsrc:
        m = os.fstat(fsrc.fileno()).st_mtime
        if seen.get(name) == m:
            return
        dst = slots[slot] if name == "last.pt" else os.path.join(remote, name)
        t0 = time.time()
        with open(dst, "wb") as fdst:  # truncate + rewrite the same Drive file
            shutil.copyfileobj(fsrc, fdst, 16 << 20)
    seen[name] = m
    if name == "last.pt":
        slot ^= 1  # flip only after a successful copy
    print(f"[sync {time.strftime('%H:%M:%S')}] {name} -> {os.path.basename(dst)} "
          f"({time.time() - t0:.0f}s)", flush=True)


while True:
    for _ in range(interval):
        if os.path.exists(stop):
            break
        time.sleep(1)
    for name in ("best.pt", "last.pt"):
        try:
            push(name)
        except Exception as e:  # e.g. Drive quota full: log, keep training locally
            print(f"[sync] {name} FAILED: {e!r}", flush=True)
    if os.path.exists(stop):
        os.remove(stop)
        print("[sync] final sync done, exiting", flush=True)
        break

Writing /content/ckpt_sync.py


## Seed study — staged

Code is pinned & verified. Run in order:

1. **Cell A** — 10-epoch base anchor (~29 min). Matched control for the sgd / lr_constant / init_xavier / init_he ablations, and an end-to-end pipeline check. Run this first.
2. **Cell B** — endpoint seed matrix: b512 + b128 × sigmoid/softmax × seeds {42,43,44} = 12 runs (~17 h). Idempotent — re-run after any preemption; finished (config, seed) pairs are skipped. Auto-shuts down the VM when the queue finishes.
3. **Cell C** — seed-grouped eval (mean ± std). Run in a **fresh session** after Cell B (Cell B releases the VM): re-run setup cells 1–7, then Cell C.

Later (Stage 3, optional): test the thesis at true small batch by setting `BASE_QUEUE` in Cell B to the b64/b32 configs.


In [ ]:
# CELL A — Stage 2: 10-epoch base anchor. RUN THIS FIRST (~29 min).
# Matched 10ep/b512/seed42/sigmoid control for the 4 recipe ablations, and a
# full pipeline check (seed -> train -> sync -> checkpoint) on the verified commit.
import os, shutil, subprocess, torch
from microclip.config import load_config

CONFIG    = "configs/base.yml"
RUN       = "base_s42_ep10"
OVERRIDES = "train.epochs=10 seed=42 run_name=base_s42_ep10 data.num_workers=8"
SYNC_EVERY_MIN = 15
STOP = "/content/ckpt_sync.stop"
LOCAL, REMOTE = f"runs/{RUN}", f"{PERSIST}/runs/{RUN}"
os.makedirs(LOCAL, exist_ok=True); os.makedirs(REMOTE, exist_ok=True)

# restore newest Drive slot if local is empty (fresh VM after a preemption)
if not os.path.exists(f"{LOCAL}/last.pt"):
    best = None
    for cand in ("last_a.pt", "last_b.pt", "last.pt"):
        p = f"{REMOTE}/{cand}"
        if not os.path.exists(p): continue
        try: step = torch.load(p, map_location="cpu", weights_only=False)["global_step"]
        except Exception: continue
        if best is None or step > best[0]: best = (step, p)
    if best:
        shutil.copyfile(best[1], f"{LOCAL}/last.pt"); print(f"restored step {best[0]}")

if os.path.exists(STOP): os.remove(STOP)
sync_proc = subprocess.Popen(
    ["python", "/content/ckpt_sync.py", LOCAL, REMOTE, str(SYNC_EVERY_MIN*60), STOP],
    stdout=open("/content/ckpt_sync.log", "a"), stderr=subprocess.STDOUT)
print("sync pid", sync_proc.pid)
get_ipython().system(f"python scripts/train.py --config {CONFIG} --set {OVERRIDES}")
open(STOP, "w").close(); sync_proc.wait()
print("base anchor done — check runs/base_s42_ep10 on Drive (expect epoch=10)")


In [ ]:
# CELL B — Stage 1: endpoint seed matrix (run after Cell A is clean).
# 4 configs x 3 seeds = 12 runs, ~17h total. b512 + b128 are the endpoints the
# thesis (small-batch robustness) and the softmax-vs-sigmoid claim rest on.
# Idempotent: a (config, seed) already finished on Drive is skipped, so you can
# re-run this cell after any preemption and it picks up where it stopped.

import os, shutil, subprocess, time, torch
from microclip.config import load_config

BASE_QUEUE = [
    "configs/sigmoid_b512.yml",
    "configs/softmax_b512.yml",
    "configs/ablations/sigmoid_b128.yml",
    "configs/ablations/softmax_b128.yml",
]
SEEDS = [42, 43, 44]
SYNC_EVERY_MIN = 15
SHUTDOWN_WHEN_DONE = True      # release the VM after the queue (stop burning units)
STOP = "/content/ckpt_sync.stop"
LOG  = f"{PERSIST}/queue_log.txt"

JOBS = []
for cfg_path in BASE_QUEUE:
    base_run = load_config(cfg_path)["run_name"]
    for s in SEEDS:
        JOBS.append((cfg_path, s, f"{base_run}_s{s}"))


def log(msg):
    line = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(line, flush=True)
    with open(LOG, "a") as f: f.write(line + "\n")


def drive_progress(remote):
    best = None
    for cand in ("last_a.pt", "last_b.pt", "last.pt"):
        p = f"{remote}/{cand}"
        if not os.path.exists(p): continue
        try: s = torch.load(p, map_location="cpu", weights_only=False)
        except Exception: continue
        if best is None or s["global_step"] > best[0]:
            best = (s["global_step"], s["epoch"], p)
    return best


def stop_sync():
    global sync_proc
    if "sync_proc" in globals() and sync_proc.poll() is None:
        open(STOP, "w").close(); sync_proc.wait()
    if os.path.exists(STOP): os.remove(STOP)


stop_sync()
for cfg_path, seed, RUN in JOBS:
    cfg = load_config(cfg_path)
    total = cfg["train"]["epochs"]
    LOCAL, REMOTE = f"runs/{RUN}", f"{PERSIST}/runs/{RUN}"
    os.makedirs(LOCAL, exist_ok=True); os.makedirs(REMOTE, exist_ok=True)

    prog = drive_progress(REMOTE)
    if prog and prog[1] >= total:
        log(f"{RUN}: already finished on Drive — skip"); continue
    if not os.path.exists(f"{LOCAL}/last.pt") and prog:
        shutil.copyfile(prog[2], f"{LOCAL}/last.pt")
        log(f"{RUN}: restored step {prog[0]} from Drive")
    if os.path.exists(f"{REMOTE}/best.pt") and not os.path.exists(f"{LOCAL}/best.pt"):
        shutil.copyfile(f"{REMOTE}/best.pt", f"{LOCAL}/best.pt")

    sync_proc = subprocess.Popen(
        ["python", "/content/ckpt_sync.py", LOCAL, REMOTE, str(SYNC_EVERY_MIN*60), STOP],
        stdout=open("/content/ckpt_sync.log", "a"), stderr=subprocess.STDOUT)
    log(f"{RUN}: training ({cfg_path}, seed={seed})")
    get_ipython().system(
        f"python scripts/train.py --config {cfg_path} "
        f"--set data.num_workers=8 seed={seed} run_name={RUN}")
    code = get_ipython().user_ns.get("_exit_code", 0)
    stop_sync()
    log(f"{RUN}: {'done' if code == 0 else f'FAILED (exit {code}) — moving on'}")

log("seed queue finished")
if SHUTDOWN_WHEN_DONE:
    from google.colab import runtime
    runtime.unassign()


In [ ]:
# CELL C — seed-grouped eval (mean +/- std). Run in a FRESH session after Cell B.
# Evaluates every run folder, groups seeds by stripping the _s<NN> suffix, and
# reports mean +/- std per metric. Single-seed groups get std = 0 (flagged n=1).

import os, re, torch, numpy as np, pandas as pd
from torchvision.datasets import CIFAR10, CIFAR100
from microclip.config import load_config
from microclip.data.coco_captions import CocoCaptions
from microclip.data.tokenizer import CaptionTokenizer
from microclip.data.transforms import build_transforms
from microclip.eval.retrieval import encode_corpus, recall_at_k
from microclip.eval.zeroshot import zeroshot_accuracy
from microclip.models.microclip import MicroCLIP

RUNS = "/content/drive/MyDrive/microclip/runs"
DEV  = "cuda"

# map every run folder -> the config it was trained with (strip _s<NN> for lookup)
CFG_BY_BASE = {
    "sigmoid_b512": "configs/sigmoid_b512.yml", "softmax_b512": "configs/softmax_b512.yml",
    "sigmoid_b256": "configs/ablations/sigmoid_b256.yml", "softmax_b256": "configs/ablations/softmax_b256.yml",
    "sigmoid_b128": "configs/ablations/sigmoid_b128.yml", "softmax_b128": "configs/ablations/softmax_b128.yml",
    "abl_vit_tiny": "configs/ablations/vit_tiny.yml", "abl_sgd": "configs/ablations/optimizer_sgd.yml",
    "abl_lr_constant": "configs/ablations/lr_constant.yml", "abl_init_xavier": "configs/ablations/init_xavier.yml",
    "abl_init_he": "configs/ablations/init_he.yml", "base_ep10": "configs/base.yml",
}

tf = build_transforms(224, train=False)
cifar = {"cifar10": CIFAR10("data/cifar", train=False, download=True, transform=tf),
         "cifar100": CIFAR100("data/cifar", train=False, download=True, transform=tf)}

per_run = {}
for run_name in sorted(os.listdir(RUNS)):
    d = f"{RUNS}/{run_name}"
    if not os.path.isdir(d) or not os.path.exists(f"{d}/best.pt"): continue
    base = re.sub(r"_s\d+", "", run_name)            # sigmoid_b128_s43 -> sigmoid_b128; base_s42_ep10 -> base_ep10
    cfg_path = CFG_BY_BASE.get(base)
    if cfg_path is None:
        print(f"skip {run_name}: no config mapping for base '{base}'"); continue
    cfg = load_config(cfg_path); dd = cfg["data"]
    tok = CaptionTokenizer(cfg["tokenizer"]["path"])
    model = MicroCLIP(cfg, vocab_size=tok.vocab_size, max_len=dd["max_text_len"])
    st = torch.load(f"{d}/best.pt", map_location="cpu", weights_only=False)
    model.load_state_dict(st["model"] if "model" in st else st)
    model = model.to(DEV).eval()
    ds = CocoCaptions(dd["root"], dd["val_images"], dd["val_ann"], tok,
                      dd["image_size"], dd["max_text_len"], train=False)
    im, tx, t2i = encode_corpus(model, ds, cfg, device=DEV)
    row = dict(recall_at_k(im, tx, t2i))
    for nm, cds in cifar.items():
        row[f"{nm}/top1"] = float(zeroshot_accuracy(model, tok, cds, cds.classes, cfg, device=DEV))
    per_run[run_name] = {k: float(v) for k, v in row.items()}
    print(run_name, {k: round(v, 4) for k, v in per_run[run_name].items()})
    del model, im, tx; torch.cuda.empty_cache()

# group by base name, report mean +/- std
groups = {}
for run_name, row in per_run.items():
    groups.setdefault(re.sub(r"_s\d+", "", run_name), []).append(row)

metrics = ["t2i/R@1","t2i/R@5","t2i/R@10","i2t/R@1","i2t/R@5","i2t/R@10","cifar10/top1","cifar100/top1"]
summary = {}
for base, rows in groups.items():
    n = len(rows)
    summary[base] = {"n": n}
    for m in metrics:
        vals = np.array([r[m] for r in rows if m in r])
        summary[base][m] = f"{vals.mean()*100:.2f} +/- {vals.std(ddof=0)*100:.2f}"

df = pd.DataFrame(summary).T
pd.DataFrame(per_run).T.to_csv(f"{RUNS}/../eval_per_run.csv")
df.to_csv(f"{RUNS}/../eval_summary.csv")
print("\n", df.to_string())
